# Car prices — training and tuning
Linear, regularized and tree models are compared with a median-price baseline. All candidates use the same train, validation and ground-truth split, with errors measured in price units.

Run the cells here to fit the selected models and write the training report, then run notebook 05 for held-out evaluation. Data preparation and feature definitions are included here, so notebooks 01–03 are not required for training.


## Choose models and run settings
Available names: DummyMedian, LinearRegression, Ridge, Lasso, ElasticNet, DecisionTree, RandomForest, GradientBoostedTreesModel, HistogramGradientBoostedTreesModel, and optional XGBoost. Set SELECTED_MODELS to a list to compare only those families; None compares every installed candidate. Tuning adds variants only from the selected families.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML

# Locate data from the notebook folder, project folder or repository root.
locations = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for location in locations
     for candidate in (location, location / "regression" / "car_price_prediction")
     if (candidate / "car_prices_dataset" / "train" / "car_price.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository or car_price_prediction folder.")
DATA_DIR = PROJECT_ROOT / "car_prices_dataset"
TRAIN_PATH = DATA_DIR / "train" / "car_price.csv"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
TARGET = "Price"
ID_COLUMN = "RowId"
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)


In [2]:
# Run settings for this notebook.
# Use None for every installed model, or enter registry names in a list:
# SELECTED_MODELS = ["DummyMedian", "Ridge", "GradientBoostedTreesModel"]
SELECTED_MODELS = None
config = {
    "random_state": 42,
    "ground_truth_fraction": 0.2,
    "validation_fraction": 0.2,
    "cv_folds": 5,
    "models": SELECTED_MODELS,
    "tune": True,
    "tune_top_n": 2,
    "overfit_ratio": 1.5,
    "underperformance_ratio": 1.25,
    "reference_year": 2016,  # Dataset snapshot year; not the current calendar year.
}
for key in ("ground_truth_fraction", "validation_fraction"):
    if not 0 < config[key] < 0.5:
        raise ValueError(f"{key} must be between 0 and 0.5")
if not isinstance(config["cv_folds"], int) or config["cv_folds"] < 2:
    raise ValueError("cv_folds must be an integer >= 2")
if not isinstance(config["tune"], bool) or not isinstance(config["tune_top_n"], int) or config["tune_top_n"] < 1:
    raise ValueError("tune must be boolean and tune_top_n must be a positive integer")
if SELECTED_MODELS is not None and (
    not isinstance(SELECTED_MODELS, list) or not SELECTED_MODELS
    or any(not isinstance(name, str) for name in SELECTED_MODELS)
    or len(SELECTED_MODELS) != len(set(SELECTED_MODELS))
):
    raise ValueError("SELECTED_MODELS must be None or a nonempty list of unique model names")
for key in ("overfit_ratio", "underperformance_ratio"):
    if not isinstance(config[key], (int, float)) or config[key] <= 1:
        raise ValueError(f"{key} must be a number greater than 1")
if not isinstance(config["reference_year"], int) or config["reference_year"] < 1886:
    raise ValueError("reference_year must be an integer >= 1886")
display(pd.Series(config, name="active_configuration").to_frame())


,active_configuration
random_state,42
ground_truth_fraction,0.2
validation_fraction,0.2
cv_folds,5
models,None
tune,True
tune_top_n,2
overfit_ratio,1.5
underperformance_ratio,1.25
reference_year,2016


## Prepare labels, deduplicate and split


In [3]:
from sklearn.model_selection import train_test_split

def prepare_data(config):
    """Assign stable source-row IDs, validate labels, deduplicate, then split."""
    raw = pd.read_csv(TRAIN_PATH)
    required = {"Brand", "Price", "Body", "Mileage", "EngineV", "Engine Type", "Registration", "Year", "Model"}
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Dataset is missing columns: {sorted(missing)}")
    if ID_COLUMN in raw:
        raise ValueError(f"Source column {ID_COLUMN} is reserved for generated row IDs")
    source_columns = raw.columns.tolist()
    raw.insert(0, ID_COLUMN, np.arange(1, len(raw) + 1))
    raw[TARGET] = pd.to_numeric(raw[TARGET], errors="coerce")
    valid_label = np.isfinite(raw[TARGET]) & (raw[TARGET] > 0)
    labeled = raw.loc[valid_label].copy()
    # Ignore generated IDs when identifying identical source records.
    duplicate = labeled.duplicated(subset=source_columns, keep="first")
    usable = labeled.loc[~duplicate].copy()
    if len(usable) < 10 or usable[TARGET].nunique() < 2:
        raise ValueError("Need at least ten labeled unique rows and two distinct positive prices")
    for key in ("ground_truth_fraction", "validation_fraction"):
        if not 0 < config[key] < 0.5:
            raise ValueError(f"{key} must be between 0 and 0.5")
    development, ground = train_test_split(usable, test_size=config["ground_truth_fraction"], random_state=config["random_state"])
    training, validation = train_test_split(development, test_size=config["validation_fraction"], random_state=config["random_state"])
    if config.get("cv_folds", 2) > len(training):
        raise ValueError("cv_folds exceeds training rows")
    assert not (set(training[ID_COLUMN]) & set(validation[ID_COLUMN])
                or set(training[ID_COLUMN]) & set(ground[ID_COLUMN])
                or set(validation[ID_COLUMN]) & set(ground[ID_COLUMN]))
    audit = {"source_rows": len(raw), "excluded_invalid_or_missing_prices": int((~valid_label).sum()),
             "excluded_exact_duplicates": int(duplicate.sum()), "usable_rows": len(usable),
             "training_rows": len(training), "validation_rows": len(validation), "ground_truth_rows": len(ground)}
    return training, validation, ground, audit


## Apply cleaning and deterministic features


In [4]:
def clean_features(X, reference_year):
    """Apply row-wise validity rules; fitted imputers handle remaining gaps."""
    data = X.drop(columns=[ID_COLUMN, TARGET, "Model"], errors="ignore").copy()
    for column in ("Mileage", "EngineV", "Year"):
        data[column] = pd.to_numeric(data[column], errors="coerce")
        data.loc[~np.isfinite(data[column]), column] = np.nan
    data.loc[data["Mileage"] < 0, "Mileage"] = np.nan
    # Treat engine volumes above 6.5 L as unknown for this dataset.
    # Treat them as unknown; retain the car instead of deleting its record.
    data.loc[(data["EngineV"] <= 0) | (data["EngineV"] > 6.5), "EngineV"] = np.nan
    data.loc[(data["Year"] < 1886) | (data["Year"] > reference_year), "Year"] = np.nan
    for column in data.select_dtypes(exclude="number").columns:
        data[column] = data[column].map(lambda value: str(value).strip() if pd.notna(value) else np.nan)
        data[column] = data[column].replace("", np.nan)
    return data

def engineer_features(X, reference_year):
    """Keep domain features deterministic and independent of prices."""
    data = clean_features(X, reference_year)
    data["AgeAtSnapshot"] = reference_year - data["Year"]
    data["LogMileage"] = np.log1p(data["Mileage"])
    # Mileage units are preserved from the source; no unit conversion is assumed.
    data["MileagePerYear"] = data["Mileage"] / (data["AgeAtSnapshot"] + 1)
    data["EngineVolumeMissing"] = data["EngineV"].isna().astype(int)
    # Age and Year encode the same linear information; keep one representation.
    return data.drop(columns="Year")


## Define preprocessing, candidate models and price-unit metrics


In [5]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor

def make_preprocessor(X: pd.DataFrame, scale_numeric: bool = True) -> ColumnTransformer:
    numeric = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical = X.select_dtypes(exclude=[np.number]).columns.tolist()
    numeric_steps = [("impute", SimpleImputer(strategy="median", keep_empty_features=True))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # scikit-learn < 1.2
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            ("categorical", Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
                ("onehot", encoder),
            ]), categorical),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

def price_model(X, estimator):
    # sklearn's estimator composition keeps learned preprocessing inside CV.
    # X contains row-wise engineered features with the identifier removed.
    trees = (GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor, DecisionTreeRegressor)
    regressor = Pipeline([
        ("preprocess", make_preprocessor(X, scale_numeric=not isinstance(estimator, trees))),
        ("model", estimator),
    ])
    return TransformedTargetRegressor(regressor=regressor, func=np.log1p, inverse_func=np.expm1)


In [6]:
def model_registry():
    models = {
        "DummyMedian": DummyRegressor(strategy="median"),
        "LinearRegression": LinearRegression(),
        "DecisionTree": DecisionTreeRegressor(random_state=config["random_state"]),
        "Ridge": Ridge(alpha=10.0),
        "Lasso": Lasso(alpha=0.0005, max_iter=20000),
        "ElasticNet": ElasticNet(alpha=0.0005, l1_ratio=0.5, max_iter=20000, random_state=config["random_state"]),
        "RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=1, max_features=0.8, random_state=config["random_state"], n_jobs=-1),
        "GradientBoostedTreesModel": GradientBoostingRegressor(n_estimators=500, learning_rate=0.03, max_depth=3, loss="huber", random_state=config["random_state"]),
        # Fit histogram boosting locally.
        "HistogramGradientBoostedTreesModel": HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=1.0, random_state=config["random_state"]),
    }
    try:
        if config["models"] is not None and "XGBoost" not in config["models"]:
            return models
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror", random_state=config["random_state"], n_jobs=-1)
    except ImportError:
        pass
    return models


In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metrics(actual, predicted):
    predicted = np.maximum(0, np.asarray(predicted))
    if not np.isfinite(predicted).all():
        raise ValueError("Nonfinite model predictions")
    actual = np.asarray(actual)
    percentage_error = np.abs(actual - predicted) / actual * 100
    return {"mape_percent": float(percentage_error.mean()),
            "within_50_percent": float(np.mean(percentage_error <= 50) * 100),
            "mae": float(mean_absolute_error(actual, predicted)),
            "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
            "rmsle": float(np.sqrt(mean_squared_error(np.log1p(actual), np.log1p(predicted)))),
            "r2": float(r2_score(actual, predicted))}

def price_rmse_scorer(model, X, y):
    return -metrics(y, model.predict(X))["rmse"]


## Reporting functions
HTML reports embed Plotly for offline viewing. RMSE is the primary comparison measure; MAE, RMSLE, R², mean absolute percentage error and the percentage within 50% of actual price provide context.


In [8]:
from html import escape
import plotly.express as px
import plotly.graph_objects as go
from plotly.io import to_html


def _table(df):
    return df.to_html(index=False, classes="dataframe", border=0, float_format=lambda x: f"{x:,.4f}")


def write_html(path: Path, title: str, intro: str, tables=None, figures=None, notes=None):
    tables = tables or []
    figures = figures or []
    notes = notes or []
    body = [f"<h1>{escape(title)}</h1><p>{escape(intro)}</p>"]
    for heading, table in tables:
        body.append(f"<h2>{escape(heading)}</h2>{_table(table)}")
    for index, (heading, figure) in enumerate(figures):
        body.append(f"<h2>{escape(heading)}</h2>{to_html(figure, full_html=False, include_plotlyjs=True if index == 0 else False)}")
    if notes:
        body.append("<h2>Evaluation notes</h2><ul>" + "".join(f"<li>{escape(n)}</li>" for n in notes) + "</ul>")
    html = """<!doctype html><html><head><meta charset='utf-8'><title>{}</title>
    <style>body{{font-family:Arial,sans-serif;max-width:1200px;margin:2rem auto;padding:0 1rem;color:#243447}}h1{{color:#12355b}}h2{{margin-top:2rem}}.dataframe{{border-collapse:collapse;width:100%}}.dataframe th,.dataframe td{{border:1px solid #ddd;padding:7px;text-align:right}}.dataframe th{{background:#eaf2f8;text-align:center}}</style>
    </head><body>{}</body></html>""".format(escape(title), "".join(body))
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(html, encoding="utf-8")


def comparison_figure(results: pd.DataFrame):
    return px.bar(results.sort_values("cv_rmse"), x="cv_rmse", y="model", orientation="h", error_x="cv_rmse_std", title="Cross-validation RMSE on Price scale")


def stage_comparison_figure(results: pd.DataFrame, metric="rmse"):
    columns = [f"{stage}_{metric}" for stage in ("train", "validation", "test") if f"{stage}_{metric}" in results]
    flags = [name for name in ("overfitting_flag", "best_model_flag", "underperforming_flag") if name in results]
    frame = results.melt(id_vars=["model", *flags], value_vars=columns, var_name="dataset", value_name=metric)
    frame["dataset"] = frame["dataset"].str.replace(f"_{metric}", "", regex=False).map({"train": "Training", "validation": "Validation", "test": "Ground-truth testing"})
    direction = "lower is better" if metric != "r2" else "higher is better"
    figure = px.bar(frame, x="model", y=metric, color="dataset", barmode="group", hover_data=flags,
                     title=f"{metric.upper()}: training vs testing ({direction})")
    figure.update_layout(xaxis_tickangle=-35)
    return figure


def actual_predicted_figure(y_true, y_pred, title):
    frame = pd.DataFrame({"actual": y_true, "predicted": y_pred})
    fig = px.scatter(frame, x="actual", y="predicted", title=title, opacity=0.7)
    limits = [min(frame.min()), max(frame.max())]
    fig.add_trace(go.Scatter(x=limits, y=limits, mode="lines", name="perfect prediction"))
    return fig


def residual_figure(y_true, y_pred):
    frame = pd.DataFrame({"predicted": y_pred, "residual": y_true - y_pred})
    fig = px.scatter(frame, x="predicted", y="residual", title="Holdout residuals", opacity=0.7)
    fig.add_hline(y=0)
    return fig


## Fit, cross-validate and optionally tune


In [9]:
import json
import time
import joblib
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, KFold

training, validation, ground, audit = prepare_data(config)
display(pd.Series(audit, name="data_preparation").to_frame())
X, y = engineer_features(training.drop(columns=TARGET), config["reference_year"]), training[TARGET]
Xv, yv = engineer_features(validation.drop(columns=TARGET), config["reference_year"]), validation[TARGET]
registry = model_registry()
if config["models"]:
    missing = set(config["models"]) - set(registry)
    if missing:
        raise ValueError(f"Unavailable model names: {sorted(missing)}")
    registry = {name: registry[name] for name in config["models"]}
cv = KFold(n_splits=config["cv_folds"], shuffle=True, random_state=config["random_state"])
records, candidates = [], {}
for name, estimator in registry.items():
    if "random_state" in estimator.get_params():
        estimator.set_params(random_state=config["random_state"])
    print(f"Comparing {name}", flush=True)
    started = time.perf_counter()
    folds = []
    for ti, vi in cv.split(X):
        model = price_model(X.iloc[ti], clone(estimator)).fit(X.iloc[ti], y.iloc[ti])
        folds.append(metrics(y.iloc[vi], model.predict(X.iloc[vi])))
    fitted = price_model(X, clone(estimator)).fit(X, y)
    train_score = metrics(y, fitted.predict(X))
    val_score = metrics(yv, fitted.predict(Xv))
    records.append({"model": name, "cv_rmse": np.mean([m["rmse"] for m in folds]),
                    "cv_rmse_std": np.std([m["rmse"] for m in folds]),
                    **{f"train_{k}": v for k, v in train_score.items()},
                    **{f"validation_{k}": v for k, v in val_score.items()},
                    "fit_seconds": time.perf_counter() - started})
    candidates[name] = fitted
base = pd.DataFrame(records).sort_values("validation_rmse")
grids = {
    "Ridge": {"alpha": [1., 10., 100.]},
    "DecisionTree": {"max_depth": [4, 8, None], "min_samples_leaf": [3, 10]},
    "Lasso": {"alpha": [0.0001, 0.0005, 0.001]},
    "ElasticNet": {"alpha": [0.0001, 0.001], "l1_ratio": [0.3, 0.7]},
    "RandomForest": {"min_samples_leaf": [1, 3]},
    "GradientBoostedTreesModel": {"max_depth": [2, 3], "learning_rate": [0.03, 0.05]},
    "HistogramGradientBoostedTreesModel": {"max_leaf_nodes": [15, 31], "l2_regularization": [1., 5.]},
    "XGBoost": {"max_depth": [2, 3], "learning_rate": [0.03, 0.05]},
}
tuning = []
if config["tune"]:
    promising = [name for name in base["model"] if name in grids][:config["tune_top_n"]]
    for name in promising:
        print(f"Tuning {name} using training CV", flush=True)
        started = time.perf_counter()
        grid = {f"regressor__model__{key}": values for key, values in grids[name].items()}
        search = GridSearchCV(price_model(X, clone(registry[name])), grid, scoring=price_rmse_scorer,
                              cv=cv, n_jobs=1, error_score="raise").fit(X, y)
        tuned = f"{name} (tuned)"
        model = search.best_estimator_
        val_score = metrics(yv, model.predict(Xv))
        scores = search.cv_results_
        train_score = metrics(y, model.predict(X))
        records.append({"model": tuned, "cv_rmse": -search.best_score_,
                        "cv_rmse_std": scores["std_test_score"][search.best_index_],
                        **{f"train_{k}": v for k, v in train_score.items()},
                        **{f"validation_{k}": v for k, v in val_score.items()}, "fit_seconds": time.perf_counter() - started})
        candidates[tuned] = model
        tuning.append({"model": tuned, "cv_rmse": -search.best_score_, "parameters": json.dumps(search.best_params_)})
results = pd.DataFrame(records).sort_values("validation_rmse").reset_index(drop=True)
results["validation_train_rmse_ratio"] = results.validation_rmse / results.train_rmse.clip(lower=1e-9)
results["overfitting_flag"] = results.validation_train_rmse_ratio > config["overfit_ratio"]

comparison = results
display(comparison)


,data_preparation
source_rows,4345
excluded_invalid_or_missing_prices,172
excluded_exact_duplicates,50
usable_rows,4123
training_rows,2638
validation_rows,660
ground_truth_rows,825


Comparing DummyMedian


Comparing LinearRegression


Comparing DecisionTree


Comparing Ridge


...

Comparing HistogramGradientBoostedTreesModel


Comparing XGBoost


Tuning DecisionTree using training CV


Tuning RandomForest using training CV


,model,cv_rmse,cv_rmse_std,train_mape_percent,train_within_50_percent,train_mae,train_rmse,train_rmsle,train_r2,validation_mape_percent,validation_within_50_percent,validation_mae,validation_rmse,validation_rmsle,validation_r2,fit_seconds,validation_train_rmse_ratio,overfitting_flag
0,DecisionTree,15561.590944,3365.154928,0.354016,100.000000,164.332216,1788.154231,0.024166,0.995129,28.894967,86.515152,4091.780855,8440.600650,0.368172,0.882977,0.076262,4.720287,True
1,RandomForest,10205.572128,3628.447021,7.027162,99.696740,1524.358528,5748.579499,0.100540,0.949655,21.322660,92.121212,3386.293999,9629.822310,0.284218,0.847679,1.269006,1.675166,True
2,RandomForest (tuned),10205.572128,3628.447021,7.027162,99.696740,1524.358528,5748.579499,0.100540,0.949655,21.322660,92.121212,3386.293999,9629.822310,0.284218,0.847679,2.161182,1.675166,True
3,XGBoost,9702.833721,3114.578493,15.553063,96.702047,3064.369059,8079.883535,0.204147,0.900540,20.715625,94.242424,3450.769573,9746.198757,0.279049,0.843975,1.099289,1.206230,False
4,HistogramGradientBoostedTreesModel,10053.532935,3408.291713,10.994736,98.711145,2213.910413,6802.911506,0.150913,0.929494,21.008191,93.636364,3430.986195,9760.474105,0.277257,0.843517,1.990154,1.434750,False
5,GradientBoostedTreesModel,10484.749401,3265.457074,16.625682,96.019712,3342.677114,10264.970698,0.224581,0.839471,20.740916,93.787879,3706.461254,12857.689417,0.286702,0.728450,5.717536,1.252579,False
6,Ridge,13882.923451,3308.911637,23.560327,89.878696,5032.086653,14198.996973,0.315020,0.692848,26.058164,90.606061,4589.127009,13732.723432,0.337007,0.690231,0.046920,0.967162,False
7,LinearRegression,13845.594346,3334.636935,23.496234,89.992418,5002.563889,14174.740197,0.314827,0.693897,26.063161,90.454545,4577.558014,13733.622266,0.337639,0.690191,0.047961,0.968880,False
8,ElasticNet,13857.597717,3329.223066,23.506668,89.916603,5013.000474,14185.321312,0.314850,0.693440,26.042194,90.606061,4582.603075,13735.321539,0.337335,0.690114,0.059918,0.968277,False
9,Lasso,13865.841352,3326.830525,23.514383,89.954511,5020.019919,14192.389643,0.314893,0.693134,26.024351,90.606061,4586.131196,13736.330687,0.337157,0.690069,0.043290,0.967866,False


## Save candidates and the training report
Each candidate is saved after fitting on training rows. Notebook 05 loads these models and their settings. A new training run removes testing outputs from the previous fits.


In [10]:
# Save everything needed to evaluate these fits in a new kernel.
import hashlib
import inspect
from datetime import datetime, timezone

def feature_signature():
    return hashlib.sha256((inspect.getsource(clean_features) + inspect.getsource(engineer_features)).encode()).hexdigest()

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
GROUND_TRUTH_DIR.mkdir(parents=True, exist_ok=True)

# Previous testing results no longer describe the new fits.
for stale in (
    ARTIFACTS_DIR / "training_testing_comparison.csv",
    ARTIFACTS_DIR / "prediction_errors.csv",
    ARTIFACTS_DIR / "price_band_comparison.csv",
    GROUND_TRUTH_DIR / "ground_truth_predictions.csv",
    GROUND_TRUTH_DIR / "ground_truth_comparison.csv",
    REPORTS_DIR / "testing_evaluation.html",
    REPORTS_DIR / "model_comparison.html",
):
    stale.unlink(missing_ok=True)

training.to_csv(TRAIN_PATH.parent / "training_data.csv", index=False)
validation.to_csv(TRAIN_PATH.parent / "validation_data.csv", index=False)
ground.drop(columns=TARGET).to_csv(GROUND_TRUTH_DIR / "ground_truth_features.csv", index=False)
ground[[ID_COLUMN, TARGET]].to_csv(GROUND_TRUTH_DIR / "ground_truth.csv", index=False)
comparison.to_csv(ARTIFACTS_DIR / "model_comparison.csv", index=False)
joblib.dump(candidates, ARTIFACTS_DIR / "candidates.joblib")
metadata = {
    "format_version": 1,
    "project": "car_price_prediction",
    "data_preparation": audit,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "config": config,
    "models": list(candidates),
    "feature_signature": feature_signature(),
    "comparison_metric": "rmse",
    "fit_rows": len(training),
    "split_rows": {"training": len(training), "validation": len(validation), "ground_truth": len(ground)},
    "tuning": tuning,
}
tracked = [
    TRAIN_PATH, TRAIN_PATH.parent / "training_data.csv", TRAIN_PATH.parent / "validation_data.csv",
    GROUND_TRUTH_DIR / "ground_truth_features.csv", GROUND_TRUTH_DIR / "ground_truth.csv",
    ARTIFACTS_DIR / "model_comparison.csv", ARTIFACTS_DIR / "candidates.joblib",
]
metadata["file_sha256"] = {
    str(path.relative_to(PROJECT_ROOT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in tracked
}
(ARTIFACTS_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
write_html(REPORTS_DIR / "training_performance.html", "Training performance",
           "Training CV, train/validation gap, and tuning use dollar RMSE throughout.",
           tables=[("Candidates", comparison), ("Tuning", pd.DataFrame(tuning))],
           figures=[("Training vs validation", stage_comparison_figure(comparison)), ("CV stability", comparison_figure(comparison))],
           notes=["Every learned transformation is fitted inside each fold.",
                  f"Overfitting risk is flagged when validation RMSE / training RMSE > {config['overfit_ratio']}.",
                  "All candidates remain available for comparison; the user decides."])
print("Training comparison saved. Run 05_testing_evaluation.ipynb next.")


Training comparison saved. Run 05_testing_evaluation.ipynb next.


## Training results
In the saved run, Decision Tree has the lowest validation RMSE at 8,441, but its training RMSE is only 1,788. The 4.72 validation/training ratio flags a large generalization gap.

Random Forest has validation RMSE of 9,630 and XGBoost 9,746. The tuned forest has the same scores as the starting forest, while tuning the decision tree raises validation RMSE to 13,900. Tuning did not improve validation performance for either selected family. Notebook 05 checks these saved fits on the held-out cars.


In [11]:
display(pd.DataFrame(metadata["tuning"]))
print("Models flagged for overfitting risk:")
display(comparison.loc[comparison.overfitting_flag, ["model", "train_rmse", "validation_rmse", "validation_train_rmse_ratio"]])
stage_comparison_figure(comparison).show()
print("Training report:", REPORTS_DIR / "training_performance.html")
display(HTML('<a href="../reports/training_performance.html" target="_blank">Training performance report</a>'))


,model,cv_rmse,parameters
0,DecisionTree (tuned),12356.048857,"{""regressor__model__max_depth"": null, ""regress..."
1,RandomForest (tuned),10205.572128,"{""regressor__model__min_samples_leaf"": 1}"


Models flagged for overfitting risk:


,model,train_rmse,validation_rmse,validation_train_rmse_ratio
0,DecisionTree,1788.154231,8440.60065,4.720287
1,RandomForest,5748.579499,9629.82231,1.675166
2,RandomForest (tuned),5748.579499,9629.82231,1.675166


Training report: /Users/blaze/Desktop/python-ml-data-science/regression/car_price_prediction/reports/training_performance.html
